In [1]:
"""
crop_centre_30mm.py
────────────────────────────────────────────────────────────────────────────
Consistent centre ±15 mm crop for BOTH windows, ALL cases.

Crop rule (identical for every case and window):
  150 pixels × 0.0003 m = 0.045 m per window
  Centre = pixel index 74.5
  ±15 mm  = ±50 px  →  keep indices 24 … 124  (101 px ≈ 30.3 mm)

World coordinates after crop:
  Window 1  →  x in [0.1303, 0.1603] m
  Window 2  →  x in [0.0628, 0.0928] m
  Gap between windows ~37.5 mm

Outputs per case  (summary_avg_std/cropped_end_30mm_centre/):
  • combined_spanwise_avg_std_cropped.csv
  • avg_temp_distribution_cropped.png   (square, full LaTeX typesetting)
────────────────────────────────────────────────────────────────────────────
"""

import os
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# ── LaTeX rendering for ALL text in the plot ──────────────────────────────────
matplotlib.rcParams.update({
    "text.usetex":         True,
    "text.latex.preamble": r"\usepackage{amsmath}",
    "font.family":         "serif",
    "font.serif":          ["Computer Modern Roman"],
    "font.size":            13,
    "axes.titlesize":       13,
    "axes.labelsize":       13,
    "legend.fontsize":      12,
    "xtick.labelsize":      12,
    "ytick.labelsize":      12,
    "axes.grid":            True,
    "grid.linestyle":       "--",
    "grid.alpha":           0.4,
    "figure.dpi":           150,
})

# ── Configuration ─────────────────────────────────────────────────────────────
ROOT       = r"D:\FCAI\plain_coupon\infared_data"
PIXEL_SIZE = 0.045 / 150          # 0.0003 m / pixel

WINDOW_CFG = {
    "window_1_spanwise_avg_std.csv": 0.1675,   # x-coordinate of index 0  [m]
    "window_2_spanwise_avg_std.csv": 0.1000,
}

SUMMARY_SUBPATH = os.path.join(
    "time_average", "average_temperature",
    "half_domain_temperature", "summary_avg_std"
)

OUT_SUBFOLDER = "cropped_end_30mm_centre"
CSV_OUT       = "combined_spanwise_avg_std_cropped.csv"
PLOT_OUT      = "avg_temp_distribution_cropped.png"

# ── Fixed crop indices: identical for every case and both windows ─────────────
CROP_LO = 24    # first kept pixel index
CROP_HI = 124   # last  kept pixel index (inclusive)
N_KEPT  = CROP_HI - CROP_LO + 1   # 101 pixels


# ═════════════════════════════════════════════════════════════════════════════
def parse_case_title(case: str) -> str:
    """Return a LaTeX-formatted string for the subplot title."""
    parts = case.split("_")
    try:
        cr_val  = parts[1]
        phi_raw = parts[3]
        if len(phi_raw) == 3:
            phi_str = phi_raw[0] + "." + phi_raw[1:]
        elif len(phi_raw) == 2:
            phi_str = "0." + phi_raw
        else:
            phi_str = phi_raw
        return (r"$\mathrm{CR} = " + cr_val +
                r"\ \mathrm{slpm},\quad \phi = " + phi_str + r"$")
    except IndexError:
        return case


# ═════════════════════════════════════════════════════════════════════════════
def process_case(case: str):
    summary_dir = os.path.join(ROOT, case, SUMMARY_SUBPATH)
    if not os.path.isdir(summary_dir):
        print(f"  [SKIP] {case}: summary dir not found")
        return

    segments = []

    for fname, origin in WINDOW_CFG.items():
        fpath = os.path.join(summary_dir, fname)
        if not os.path.isfile(fpath):
            print(f"  [WARN] {case}: file not found -> {fname}")
            continue

        df = pd.read_csv(fpath, header=0,
                         names=["col_index", "spanwise_avg_C", "spanwise_std_C"])

        # ── Fixed centre crop ──────────────────────────────────────────────────
        mask = (df["col_index"] >= CROP_LO) & (df["col_index"] <= CROP_HI)
        df_c = df[mask].copy()

        if len(df_c) != N_KEPT:
            print(f"  [WARN] {case}/{fname}: expected {N_KEPT} rows, "
                  f"got {len(df_c)}")

        # World coordinate (reverse-streamwise)
        df_c["x_world_m"] = origin - df_c["col_index"].astype(float) * PIXEL_SIZE

        seg = df_c[["x_world_m", "spanwise_avg_C", "spanwise_std_C"]].copy()
        segments.append(seg)
        print(f"    {fname}: kept idx [{CROP_LO}..{CROP_HI}] ({N_KEPT} px)"
              f"  x in [{seg['x_world_m'].min():.4f}, "
              f"{seg['x_world_m'].max():.4f}] m")

    if not segments:
        print(f"  [SKIP] {case}: no data loaded")
        return

    combined = pd.concat(segments, ignore_index=True)
    combined.sort_values("x_world_m", inplace=True)
    combined.reset_index(drop=True, inplace=True)

    # ── Save CSV ───────────────────────────────────────────────────────────────
    out_dir = os.path.join(summary_dir, OUT_SUBFOLDER)
    os.makedirs(out_dir, exist_ok=True)

    csv_path = os.path.join(out_dir, CSV_OUT)
    combined.to_csv(csv_path, index=False)
    print(f"    CSV  -> {csv_path}")

    # ── Detect inter-window gap ────────────────────────────────────────────────
    x_arr    = combined["x_world_m"].values
    T_avg    = combined["spanwise_avg_C"].values
    T_std    = combined["spanwise_std_C"].values
    x_mm     = x_arr * 1e3
    dx       = np.diff(x_arr)
    gap_idxs = np.where(dx > 5 * PIXEL_SIZE)[0]

    # Continuous drawing segments
    split_pts  = [0] + [g + 1 for g in gap_idxs] + [len(x_arr)]
    seg_slices = [slice(split_pts[i], split_pts[i + 1])
                  for i in range(len(split_pts) - 1)]

    # ── Square figure with full LaTeX typography ───────────────────────────────
    fig, ax = plt.subplots(figsize=(7, 7))

    seg_colors   = ["#0D47A1", "#1565C0"]
    sigma_colors = ["#2196F3", "#42A5F5"]
    avg_labelled   = False
    sigma_labelled = False

    for k, sl in enumerate(seg_slices):
        xk  = x_mm[sl]
        Tk  = T_avg[sl]
        Sk  = T_std[sl]
        c_line = seg_colors[k % len(seg_colors)]
        c_band = sigma_colors[k % len(sigma_colors)]

        ax.fill_between(
            xk, Tk - Sk, Tk + Sk,
            alpha=0.22, color=c_band,
            label=(r"$\pm 1\sigma$" if not sigma_labelled else "_nolegend_"),
        )
        sigma_labelled = True

        ax.plot(
            xk, Tk, color=c_line, linewidth=1.8,
            label=(r"$\bar{T}$ (spanwise avg)" if not avg_labelled else "_nolegend_"),
        )
        avg_labelled = True

    # Dashed bridge across gap(s)
    for g in gap_idxs:
        xb = [x_mm[g], x_mm[g + 1]]
        Tb = [T_avg[g], T_avg[g + 1]]
        ax.plot(xb, Tb, color="#888888", linewidth=1.4, linestyle="--", zorder=3,
                label=(r"Gap (no data)" if g == gap_idxs[0] else "_nolegend_"))

        gap_mm = x_mm[g + 1] - x_mm[g]
        ax.annotate(
            r"$\mathrm{gap}\approx " + f"{gap_mm:.1f}" + r"\,\mathrm{mm}$",
            xy=((xb[0] + xb[1]) / 2, (Tb[0] + Tb[1]) / 2),
            xytext=(0, 14), textcoords="offset points",
            ha="center", fontsize=11, color="#555555",
        )

    # Axis labels and title — all LaTeX
    ax.set_xlabel(r"Streamwise distance (mm)")
    ax.set_ylabel(r"Temperature ($^\circ$C)")
    ax.set_title(
        r"\textbf{Spanwise-Averaged Temperature}" + "\n" + parse_case_title(case),
        pad=10,
    )
    ax.legend(loc="best", framealpha=0.85)
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator(5))
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator(5))
    ax.tick_params(which="minor", length=3, color="gray")

    # # Secondary top axis in metres
    # ax2 = ax.twiny()
    # ax2.set_xlim(ax.get_xlim())
    # ax2.set_xlabel(r"Streamwise distance (m)", labelpad=6)
    # xticks_mm = ax.get_xticks()
    # ax2.set_xticks(xticks_mm)
    # ax2.set_xticklabels([r"$" + f"{v / 1e3:.4f}" + r"$" for v in xticks_mm])

    fig.tight_layout()
    png_path = os.path.join(out_dir, PLOT_OUT)
    fig.savefig(png_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"    Plot -> {png_path}\n")


# ═════════════════════════════════════════════════════════════════════════════
if __name__ == "__main__":
    case_dirs = sorted([
        d for d in os.listdir(ROOT)
        if os.path.isdir(os.path.join(ROOT, d)) and d.startswith("cr_")
    ])

    print(f"Found {len(case_dirs)} case(s): {case_dirs}\n")
    print(f"Fixed crop rule (all cases, both windows):")
    print(f"  Indices kept : {CROP_LO}..{CROP_HI}  "
          f"({N_KEPT} px = {N_KEPT * PIXEL_SIZE * 1e3:.1f} mm)")
    print(f"  Win 1 x range: [{0.1675 - CROP_HI*PIXEL_SIZE:.4f}, "
          f"{0.1675 - CROP_LO*PIXEL_SIZE:.4f}] m")
    print(f"  Win 2 x range: [{0.1000 - CROP_HI*PIXEL_SIZE:.4f}, "
          f"{0.1000 - CROP_LO*PIXEL_SIZE:.4f}] m\n")

    for case in case_dirs:
        print(f"-- {case}")
        process_case(case)

    print("Done.")

Found 15 case(s): ['cr_1400_phi_085', 'cr_1400_phi_09', 'cr_1400_phi_10', 'cr_1400_phi_118', 'cr_1400_phi_137', 'cr_2300_phi_085', 'cr_2300_phi_09', 'cr_2300_phi_10', 'cr_2300_phi_118', 'cr_2300_phi_137', 'cr_500_phi_085', 'cr_500_phi_09', 'cr_500_phi_10', 'cr_500_phi_118', 'cr_500_phi_137']

Fixed crop rule (all cases, both windows):
  Indices kept : 24..124  (101 px = 30.3 mm)
  Win 1 x range: [0.1303, 0.1603] m
  Win 2 x range: [0.0628, 0.0928] m

-- cr_1400_phi_085
    window_1_spanwise_avg_std.csv: kept idx [24..124] (101 px)  x in [0.1303, 0.1603] m
    window_2_spanwise_avg_std.csv: kept idx [24..124] (101 px)  x in [0.0628, 0.0928] m
    CSV  -> D:\FCAI\plain_coupon\infared_data\cr_1400_phi_085\time_average\average_temperature\half_domain_temperature\summary_avg_std\cropped_end_30mm_centre\combined_spanwise_avg_std_cropped.csv
    Plot -> D:\FCAI\plain_coupon\infared_data\cr_1400_phi_085\time_average\average_temperature\half_domain_temperature\summary_avg_std\cropped_end_30mm_c